In [1]:
%pip install kagglehub


   ---------------------------------------- 0/8 [urllib3]
   ---------------------------------------- 0/8 [urllib3]
   ---------------------------------------- 0/8 [urllib3]
   ----- ---------------------------------- 1/8 [tqdm]
   ----- ---------------------------------- 1/8 [tqdm]
   ---------- ----------------------------- 2/8 [pyyaml]
   ---------- ----------------------------- 2/8 [pyyaml]
   -------------------- ------------------- 4/8 [charset_normalizer]
   -------------------- ------------------- 4/8 [charset_normalizer]
   ------------------------------ --------- 6/8 [requests]
   ----------------------------------- ---- 7/8 [kagglehub]
   ----------------------------------- ---- 7/8 [kagglehub]
   ----------------------------------- ---- 7/8 [kagglehub]
   ---------------------------------------- 8/8 [kagglehub]

Note: you may need to restart the kernel to use updated packages.


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fabdelja/autism-screening-for-toddlers")

c:\Users\vpaga\OneDrive\Documents\autism-llm-support\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 81.7k/81.7k [00:00<00:00, 54.9MB/s]

Extracting files...


In [5]:
%pip install pandas

   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   -------------------------------------- - 10.5/11.0 MB 55.0 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 47.3 MB/s  0:00:00
   ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
   ---------------------------------------  12.6/12.8 MB 65.4 MB/s eta 0:00:01
   ---------------------------------------- 12.8/12.8 MB 56.9 MB/s  0:00:00

   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   ---------- ----------------------------- 1/4 [tzdata]
   ---------- ----------------------------- 1/4 [tzdata]
   ---------- ----------------------------- 1/4 [tzdata]
   -------------------- ------------------- 2/4 [numpy]
   -------------------- ------------------- 2/4 [numpy]
   -------------------- ------------------- 2/4 [numpy]
   -------------------- ------------------- 2/4 [numpy]
   -------------------- ------------------- 

In [8]:
%pip install transformers peft datasets

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ----------------------------- ---------- 8.4/11.3 MB 46.8 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 43.4 MB/s  0:00:00
   ---------------------------------------- 0.0/561.5 kB ? eta -:--:--
   ---------------------------------------- 561.5/561.5 kB 23.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 55.8 MB/s  0:00:00
   ---------------------------------------- 0.0/26.1 MB ? eta -:--:--
   ------------------ --------------------- 12.1/26.1 MB 59.8 MB/s eta 0:00:01
   -------------------------------------- - 25.4/26.1 MB 62.6 MB/s eta 0:00:01
   ---------------------------------------- 26.1/26.1 MB 55.4 MB/s  0:00:00
   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
   - -------------------------------------- 9.7/241.3 MB 46.2 MB/s eta 0:00:06
   ---- ----------------------

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading checkpoint shards: 100%|██████████| 2/2 [00:17<00:00,  8.92s/it]
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."<|eot_id|>


In [3]:
import pandas as pd
import json

# Load datasets
df1 = pd.read_csv("Autism_Screening_Data_Combined.csv")
df2 = pd.read_csv("Toddler Autism dataset July 2018.csv")

# Merge
df = pd.concat([df1, df2], ignore_index=True)

# Drop Score if exists
if "Score" in df.columns:
    df = df.drop(columns=["Score"])

# Map A1–A10 to text questions
question_map = {
    "A1": "Does your child look at you when you call his/her name?",
    "A2": "How easy is it for you to get eye contact with your child?",
    "A3": "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)?",
    "A4": "Does your child point to share interest with you (e.g. pointing at an interesting sight)?",
    "A5": "Does your child pretend (e.g. care for dolls, talk on a toy phone)?",
    "A6": "Does your child follow where you’re looking?",
    "A7": "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them?",
    "A8": "Would you describe your child’s first words as clear and meaningful?",
    "A9": "Does your child use simple gestures (e.g. wave goodbye)?",
    "A10": "Does your child stare at nothing with no apparent purpose?"
}

# Write JSONL
with open("train.jsonl", "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        qa_pairs = []
        for col, question in question_map.items():
            qa_pairs.append(f"{question} Answer: {str(row[col])}")

        extra_info = f"Age: {str(row['Age'])}, Sex: {str(row['Sex'])}, Jaundice: {str(row['Jauundice'])}, Family history of ASD: {str(row['Family_ASD'])}"

        user_prompt = "Patient screening info:\n" + "\n".join(qa_pairs) + f"\n{extra_info}\n\nDoes this child have autism risk? Answer YES or NO."
        assistant_answer = str(row["Class"])

        sample = {
            "messages": [
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": assistant_answer}
            ]
        }

        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("✅ train.jsonl created successfully")


✅ train.jsonl created successfully


In [4]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="train.jsonl", split="train")
print(dataset[0])


Generating train split: 7129 examples [00:00, 85044.90 examples/s]

{'messages': [{'role': 'user', 'content': 'Patient screening info:\nDoes your child look at you when you call his/her name? Answer: 1\nHow easy is it for you to get eye contact with your child? Answer: 1\nDoes your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 0\nDoes your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 1\nDoes your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 0\nDoes your child follow where you’re looking? Answer: 0\nIf you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 1\nWould you describe your child’s first words as clear and meaningful? Answer: 1\nDoes your child use simple gestures (e.g. wave goodbye)? Answer: 0\nDoes your child stare at nothing with no apparent purpose? Answer: 0\nAge: 15.0, Sex: m, Jaundice: no, Family history of ASD: no\n\nDoes this child have autism risk? Answer YES or NO.'

In [5]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="train.jsonl", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset["train"][0])
print(dataset["test"][0])


{'messages': [{'role': 'user', 'content': 'Patient screening info:\nDoes your child look at you when you call his/her name? Answer: 0\nHow easy is it for you to get eye contact with your child? Answer: 1\nDoes your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 0\nDoes your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 0\nDoes your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 0\nDoes your child follow where you’re looking? Answer: 0\nIf you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 0\nWould you describe your child’s first words as clear and meaningful? Answer: 0\nDoes your child use simple gestures (e.g. wave goodbye)? Answer: 0\nDoes your child stare at nothing with no apparent purpose? Answer: 0\nAge: 20.0, Sex: m, Jaundice: no, Family history of ASD: no\n\nDoes this child have autism risk? Answer YES or NO.'

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer

model_name = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

training_args = TrainingArguments(
    output_dir="./llama-autism",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    num_train_epochs=3,
    do_eval=True,                # instead of evaluation_strategy
    eval_steps=50,
    save_steps=50,
    logging_steps=10,
    save_total_limit=2,
    bf16=True,                   # if you don’t have A100/H100, switch to fp16=True
    report_to=[]                 # [] avoids wandb/tensorboard issues
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="messages",   # very important
    max_seq_length=512,
    args=training_args,
)

trainer.train()


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

: 

In [6]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig

# ===== Config =====
model_name = "meta-llama/Llama-3.2-1B-Instruct"
train_jsonl = "train.jsonl"
output_dir = "./llama-autism-cpu"
num_train_epochs = 1

# ===== Load dataset =====
dataset = load_dataset("json", data_files=train_jsonl, split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = dataset["train"]
eval_ds = dataset["test"]

# ===== Preprocess dataset =====
def preprocess(example):
    user_msg = example['messages'][0]['content']
    assistant_msg = example['messages'][1]['content']
    return {"text": f"{user_msg}\n{assistant_msg}"}

train_ds = train_ds.map(preprocess)
eval_ds = eval_ds.map(preprocess)

# ===== Tokenizer & Model =====
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map={"": "cpu"})

# ===== LoRA config =====
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# ===== Training args =====
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=num_train_epochs,
    learning_rate=2e-4,
    fp16=False,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    report_to=[],
    do_eval=True
)

# ===== Trainer =====
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
    args=training_args
)

# ===== Train =====
trainer.train()

# ===== Save model =====
trainer.save_state()
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ Training finished. Model + tokenizer saved to:", output_dir)


Truncating eval dataset: 100%|██████████| 713/713 [00:00<00:00, 30993.25 examples/s]
c:\Users\vpaga\OneDrive\Documents\autism-llm-support\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
20,1.595600
40,0.160500
60,0.095300
80,0.094500
100,0.094300
120,0.092600
140,0.091200
160,0.091500
180,0.091000
200,0.091500


✅ Training finished. Model + tokenizer saved to: ./llama-autism-cpu


In [14]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   --------------------- ------------------ 4.7/8.7 MB 24.2 MB/s eta 0:00:01
   ------------------------------- -------- 6.8/8.7 MB 18.7 MB/s eta 0:00:01
   ------------------------------------- -- 8.1/8.7 MB 13.6 MB/s eta 0:00:01
   ---------------------------------------- 8.7/8.7 MB 11.4 MB/s  0:00:00
   ---------------------------------------- 0.0/38.5 MB ? eta -:--:--
   - -------------------------------------- 1.0/38.5 MB 12.6 MB/s eta 0:00:03
   --- ------------------------------------ 3.1/38.5 MB 7.3 MB/s eta 0:00:05
   ---- ----------------------------------- 4.2/38.5 MB 6.3 MB/s eta 0:00:06
   ----- ---------------------------------- 5.5/38.5 MB 6.7 MB/s eta 0:00:05
   ------ --------------------------------- 6.3/38.5 MB 6.0 MB/s eta 0:00:06
   ------- -------------------------------- 7.1/38.5 MB 5.6 MB/s eta 0:00:06
   ------- -------------------------------- 7.6/38.5 MB 5.4 MB/s eta 0:00:06
   -------- ----

In [17]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import os

# ===== Config =====
model_name = "meta-llama/Llama-3.2-1B-Instruct"
output_dir = "./llama-autism-cpu"
device = "cpu"

# ===== Synthetic test dataset =====
test_data = [
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 0\n"
                    "How easy is it for you to get eye contact with your child? Answer: 0\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 0\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 0\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 0\n"
                    "Does your child follow where you’re looking? Answer: 0\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 0\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 0\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 0\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 1\n"
                    "Age: 18.0, Sex: f, Jaundice: yes, Family history of ASD: yes\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "YES"}
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 1\n"
                    "How easy is it for you to get eye contact with your child? Answer: 1\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 1\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 1\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 1\n"
                    "Does your child follow where you’re looking? Answer: 1\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 1\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 1\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 1\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 0\n"
                    "Age: 24.0, Sex: m, Jaundice: no, Family history of ASD: no\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "NO"}
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 0\n"
                    "How easy is it for you to get eye contact with your child? Answer: 0\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 0\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 0\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 1\n"
                    "Does your child follow where you’re looking? Answer: 0\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 0\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 0\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 0\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 1\n"
                    "Age: 20.0, Sex: f, Jaundice: no, Family history of ASD: yes\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "YES"}
        ]
    }
]

# Convert to Dataset
test_dataset = Dataset.from_list(test_data)

# Preprocess dataset
def preprocess(example):
    user_msg = example['messages'][0]['content']
    assistant_msg = example['messages'][1]['content']
    return {"text": f"{user_msg}\n{assistant_msg}", "label": assistant_msg}

test_dataset = test_dataset.map(preprocess)

# ===== Load tokenizer and model =====
try:
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cpu",
        low_cpu_mem_usage=True,
        torch_dtype=torch.float32
    )
    model = PeftModel.from_pretrained(
        base_model,
        output_dir,
        device_map="cpu",
        is_trainable=False
    )
    model.eval()
except Exception as e:
    print(f"Error loading model or tokenizer: {e}")
    exit(1)

# ===== Generate predictions =====
def generate_prediction(text):
    try:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
        outputs = model.generate(**inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)
        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract YES or NO by searching for these words in the output
        prediction = prediction.upper()
        if "YES" in prediction:
            return "YES"
        elif "NO" in prediction:
            return "NO"
        else:
            return "UNKNOWN"
    except Exception as e:
        print(f"Error generating prediction: {e}")
        return "UNKNOWN"

predictions = []
true_labels = []

for i, example in enumerate(test_dataset):
    user_text = example["text"].split("\nNO")[0].split("\nYES")[0]  # Extract user input part
    pred = generate_prediction(user_text)
    predictions.append(pred)
    true_labels.append(example["label"])
    print(f"Example {i+1}: Prediction = {pred}, True Label = {example['label']}")

# ===== Evaluate =====
try:
    # Filter out "UNKNOWN" predictions to avoid errors in classification_report
    valid_pairs = [(pred, label) for pred, label in zip(predictions, true_labels) if pred != "UNKNOWN"]
    if not valid_pairs:
        print("No valid predictions (all 'UNKNOWN'). Cannot compute metrics.")
        exit(1)
    valid_predictions, valid_true_labels = zip(*valid_pairs)
    accuracy = accuracy_score(valid_true_labels, valid_predictions)
    report = classification_report(valid_true_labels, valid_predictions, target_names=["NO", "YES"], output_dict=True)
except Exception as e:
    print(f"Error computing metrics: {e}")
    exit(1)

# ===== Print results =====
print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(pd.DataFrame(report).transpose())

# Save results to a file
os.makedirs(output_dir, exist_ok=True)
with open(f"{output_dir}/test_results.txt", "w") as f:
    f.write(f"Accuracy: {accuracy:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(pd.DataFrame(report).transpose().to_string())

print(f"✅ Evaluation complete. Results saved to {output_dir}/test_results.txt")

Map: 100%|██████████| 3/3 [00:00<00:00, 167.93 examples/s]


Example 1: Prediction = YES, True Label = YES
Example 2: Prediction = YES, True Label = NO
Example 3: Prediction = YES, True Label = YES

Accuracy: 0.6667

Classification Report:
              precision    recall  f1-score   support
NO             0.000000  0.000000  0.000000  1.000000
YES            0.666667  1.000000  0.800000  2.000000
accuracy       0.666667  0.666667  0.666667  0.666667
macro avg      0.333333  0.500000  0.400000  3.000000
weighted avg   0.444444  0.666667  0.533333  3.000000
✅ Evaluation complete. Results saved to ./llama-autism-cpu/test_results.txt


c:\Users\vpaga\OneDrive\Documents\autism-llm-support\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\vpaga\OneDrive\Documents\autism-llm-support\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\vpaga\OneDrive\Documents\autism-llm-support\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

In [20]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import os

# ===== Config =====
model_name = "meta-llama/Llama-3.2-1B-Instruct"
output_dir = "./llama-autism-cpu"
device = "cpu"

# ===== Synthetic test dataset (10 examples) =====
test_data = [
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 0\n"
                    "How easy is it for you to get eye contact with your child? Answer: 0\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 0\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 0\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 0\n"
                    "Does your child follow where you’re looking? Answer: 0\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 0\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 0\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 0\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 1\n"
                    "Age: 18.0, Sex: f, Jaundice: yes, Family history of ASD: yes\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "YES"}
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 1\n"
                    "How easy is it for you to get eye contact with your child? Answer: 1\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 1\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 1\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 1\n"
                    "Does your child follow where you’re looking? Answer: 1\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 1\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 1\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 1\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 0\n"
                    "Age: 24.0, Sex: m, Jaundice: no, Family history of ASD: no\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "NO"}
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 0\n"
                    "How easy is it for you to get eye contact with your child? Answer: 0\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 0\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 0\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 1\n"
                    "Does your child follow where you’re looking? Answer: 0\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 0\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 0\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 0\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 1\n"
                    "Age: 20.0, Sex: f, Jaundice: no, Family history of ASD: yes\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "YES"}
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 1\n"
                    "How easy is it for you to get eye contact with your child? Answer: 1\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 1\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 1\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 0\n"
                    "Does your child follow where you’re looking? Answer: 1\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 1\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 1\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 1\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 0\n"
                    "Age: 22.0, Sex: m, Jaundice: no, Family history of ASD: no\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "NO"}
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 0\n"
                    "How easy is it for you to get eye contact with your child? Answer: 0\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 1\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 0\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 0\n"
                    "Does your child follow where you’re looking? Answer: 0\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 0\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 0\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 0\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 1\n"
                    "Age: 16.0, Sex: m, Jaundice: yes, Family history of ASD: no\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "YES"}
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 1\n"
                    "How easy is it for you to get eye contact with your child? Answer: 1\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 0\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 1\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 1\n"
                    "Does your child follow where you’re looking? Answer: 1\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 1\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 1\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 1\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 0\n"
                    "Age: 19.0, Sex: f, Jaundice: no, Family history of ASD: no\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "NO"}
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 0\n"
                    "How easy is it for you to get eye contact with your child? Answer: 0\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 0\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 0\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 0\n"
                    "Does your child follow where you’re looking? Answer: 0\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 1\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 0\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 0\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 1\n"
                    "Age: 17.0, Sex: m, Jaundice: yes, Family history of ASD: yes\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "YES"}
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 1\n"
                    "How easy is it for you to get eye contact with your child? Answer: 1\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 1\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 1\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 1\n"
                    "Does your child follow where you’re looking? Answer: 0\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 1\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 1\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 1\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 0\n"
                    "Age: 21.0, Sex: f, Jaundice: no, Family history of ASD: no\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "NO"}
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 0\n"
                    "How easy is it for you to get eye contact with your child? Answer: 0\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 0\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 1\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 0\n"
                    "Does your child follow where you’re looking? Answer: 0\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 0\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 0\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 0\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 1\n"
                    "Age: 23.0, Sex: m, Jaundice: yes, Family history of ASD: yes\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "YES"}
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Patient screening info:\n"
                    "Does your child look at you when you call his/her name? Answer: 1\n"
                    "How easy is it for you to get eye contact with your child? Answer: 1\n"
                    "Does your child point to indicate that s/he wants something (e.g. a toy that is out of reach)? Answer: 1\n"
                    "Does your child point to share interest with you (e.g. pointing at an interesting sight)? Answer: 0\n"
                    "Does your child pretend (e.g. care for dolls, talk on a toy phone)? Answer: 1\n"
                    "Does your child follow where you’re looking? Answer: 1\n"
                    "If you or someone else in the family is visibly upset, does your child show signs of wanting to comfort them? Answer: 1\n"
                    "Would you describe your child’s first words as clear and meaningful? Answer: 1\n"
                    "Does your child use simple gestures (e.g. wave goodbye)? Answer: 1\n"
                    "Does your child stare at nothing with no apparent purpose? Answer: 0\n"
                    "Age: 20.0, Sex: m, Jaundice: no, Family history of ASD: no\n\n"
                    "Does this child have autism risk? Answer YES or NO."
                )
            },
            {"role": "assistant", "content": "NO"}
        ]
    }
]

# Convert to Dataset
test_dataset = Dataset.from_list(test_data)

# Preprocess dataset
def preprocess(example):
    user_msg = example['messages'][0]['content']
    assistant_msg = example['messages'][1]['content']
    return {"text": f"{user_msg}\n{assistant_msg}", "label": assistant_msg}

test_dataset = test_dataset.map(preprocess)

# ===== Load tokenizer and model =====
try:
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cpu",
        low_cpu_mem_usage=True,
        torch_dtype=torch.float32
    )
    model = PeftModel.from_pretrained(
        base_model,
        output_dir,
        device_map="cpu",
        is_trainable=False
    )
    model.eval()
except Exception as e:
    print(f"Error loading model or tokenizer: {e}")
    exit(1)

# ===== Generate predictions =====
def generate_prediction(text):
    try:
        # Use a system prompt to enforce clear YES/NO output
        system_prompt = (
            "You are a medical assistant. Based on the provided patient screening information, "
            "predict whether the child has an autism risk. Respond with only 'YES' or 'NO'."
        )
        prompt = f"{system_prompt}\n\n{text}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
        input_len = inputs["input_ids"].shape[1]
        outputs = model.generate(**inputs, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
        generated_tokens = outputs[0][input_len:]
        raw_generated = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        print(f"Raw generated output: {raw_generated}")
        # Extract YES or NO from the generated text
        generated_upper = raw_generated.upper()
        if "YES" in generated_upper and "NO" not in generated_upper:
            return "YES"
        elif "NO" in generated_upper and "YES" not in generated_upper:
            return "NO"
        elif "YES" in generated_upper and "NO" in generated_upper:
            # If both, take the first occurrence
            yes_pos = generated_upper.find("YES")
            no_pos = generated_upper.find("NO")
            return "YES" if yes_pos < no_pos else "NO"
        else:
            return "UNKNOWN"
    except Exception as e:
        print(f"Error generating prediction: {e}")
        return "UNKNOWN"

predictions = []
true_labels = []

for i, example in enumerate(test_dataset):
    user_text = example["text"].split("\nNO")[0].split("\nYES")[0]  # Extract user input part
    pred = generate_prediction(user_text)
    predictions.append(pred)
    true_labels.append(example["label"])
    print(f"Example {i+1}: Prediction = {pred}, True Label = {example['label']}")

# ===== Evaluate =====
try:
    # Filter out "UNKNOWN" predictions to avoid errors in classification_report
    valid_pairs = [(pred, label) for pred, label in zip(predictions, true_labels) if pred != "UNKNOWN"]
    if not valid_pairs:
        print("No valid predictions (all 'UNKNOWN'). Cannot compute metrics.")
        exit(1)
    valid_predictions, valid_true_labels = zip(*valid_pairs)
    accuracy = accuracy_score(valid_true_labels, valid_predictions)
    report = classification_report(
        valid_true_labels,
        valid_predictions,
        target_names=["NO", "YES"],
        output_dict=True,
        zero_division=0
    )
except Exception as e:
    print(f"Error computing metrics: {e}")
    exit(1)

# ===== Print results =====
print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(pd.DataFrame(report).transpose())

# Save results to a file
os.makedirs(output_dir, exist_ok=True)
with open(f"{output_dir}/test_results.txt", "w") as f:
    f.write(f"Accuracy: {accuracy:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(pd.DataFrame(report).transpose().to_string())

print(f"✅ Evaluation complete. Results saved to {output_dir}/test_results.txt")

Map: 100%|██████████| 10/10 [00:00<00:00, 305.34 examples/s]


Raw generated output: NO
Example 1: Prediction = NO, True Label = YES
Raw generated output: YES
Example 2: Prediction = YES, True Label = NO
Raw generated output: NO
Example 3: Prediction = NO, True Label = YES
Raw generated output: NO
Example 4: Prediction = NO, True Label = NO
Raw generated output: NO
Example 5: Prediction = NO, True Label = YES
Raw generated output: NO
Example 6: Prediction = NO, True Label = NO
Raw generated output: NO
Example 7: Prediction = NO, True Label = YES
Raw generated output: NO
Example 8: Prediction = NO, True Label = NO
Raw generated output: NO
Example 9: Prediction = NO, True Label = YES
Raw generated output: NO
Example 10: Prediction = NO, True Label = NO

Accuracy: 0.4000

Classification Report:
              precision  recall  f1-score  support
NO             0.444444     0.8  0.571429      5.0
YES            0.000000     0.0  0.000000      5.0
accuracy       0.400000     0.4  0.400000      0.4
macro avg      0.222222     0.4  0.285714     10.0
weigh